In [76]:
import sqlite3
import pandas as pd 

connection = sqlite3.connect("../data/movies.db")

movies = pd.read_sql_query("SELECT * FROM movies", connection)
genres = pd.read_sql_query("SELECT * FROM genres", connection)
movie_genres = pd.read_sql_query("SELECT * FROM movie_genres", connection)
people = pd.read_sql_query("SELECT * FROM people", connection)
movie_credits = pd.read_sql_query("SELECT * FROM movie_credits", connection)

connection.close()

In [77]:
links = pd.read_csv("../data/raw/ml-latest-small/links.csv")
all_ids = set(links['tmdbId'].dropna())
saved_ids = set(movies['tmdb_id'].dropna())
missing_ids = all_ids - saved_ids
missing_movies_ids =  links[links['tmdbId'].isin(missing_ids)]

In [78]:
len(saved_ids)/len(all_ids)

0.9883900133566218

In [79]:
movies[movies[['title', 'release_date']].duplicated(keep=False)].sort_values('title')

,tmdb_id,title,release_date,runtime,budget,revenue,vote_average,vote_count


In [80]:
movie_with_genres = movie_genres.merge(
    movies, how='left', on='tmdb_id').merge(
        genres, how='left', right_on='id', left_on='genre_id')

genre_stats = movie_with_genres.groupby('genre').apply(
    lambda x: pd.Series({
        'average_rating': x['vote_average'].mean(),
        'weighted_rating': (
            (x['vote_average'] * x['vote_count']).sum()
            / x['vote_count'].sum()
        ),
        'review_count': x['vote_count'].sum(),
        'movie_count': x['tmdb_id'].nunique()
    })
).round(2)
genre_stats.sort_values('weighted_rating', ascending=False)

,average_rating,weighted_rating,review_count,movie_count
genre,,,,
War,6.93,7.46,577655.0,332.0
History,6.95,7.38,650324.0,445.0
Documentary,7.05,7.31,86680.0,414.0
Western,6.71,7.30,273965.0,170.0
Drama,6.73,7.29,6549879.0,4474.0
Animation,6.99,7.26,1682640.0,598.0
Music,6.71,7.19,283300.0,348.0
Crime,6.57,7.15,2811366.0,1482.0
Mystery,6.58,7.01,1413009.0,698.0


In [83]:
movie_with_cast = movies.merge(
    movie_credits, how='left', on='tmdb_id').merge(
        people, how='left', on='person_id')

cast_stats = movie_with_cast.groupby(['person_id', 'role']).agg(
    average_rating=('vote_average', 'mean'),
    review_count=('vote_count', 'sum'),
    movie_count=('tmdb_id', 'count')
).round(2).reset_index()
cast_stats = people.merge(cast_stats, how='right', on='person_id').set_index('person_id')

directors = cast_stats[cast_stats['role'] == 'Director'].sort_values('review_count', ascending=False)
actors = cast_stats[cast_stats['role'] == 'Actor'].sort_values('review_count', ascending=False)
display(actors.head())
display(directors.head())

,name,role,average_rating,review_count,movie_count
person_id,,,,,
85,Johnny Depp,Actor,6.69,285508,48
3223,Robert Downey Jr.,Actor,6.60,280738,41
31,Tom Hanks,Actor,6.92,280724,42
6193,Leonardo DiCaprio,Actor,7.26,278810,25
287,Brad Pitt,Actor,7.05,274528,33


,name,role,average_rating,review_count,movie_count
person_id,,,,,
488,Steven Spielberg,Director,7.21,260433,32
525,Christopher Nolan,Director,7.88,230304,10
138,Quentin Tarantino,Director,7.87,165854,9
1032,Martin Scorsese,Director,7.35,155709,25
578,Ridley Scott,Director,6.83,154365,23


In [82]:
movies['profit'] = movies['revenue'] - movies['budget']
movies = movies[movies['revenue'] != 0]
pd.concat([
    movies.sort_values('profit', ascending=False).head(),
    movies.sort_values('profit', ascending=False).tail()
]).style.format({
    'budget': '{:,.0f}',
    'revenue': '{:,.0f}',
    'profit': '{:,.0f}'
})

,tmdb_id,title,release_date,runtime,budget,revenue,vote_average,vote_count,profit
5326,19995,Avatar,2009-12-16,162,"237,000,000","2,923,706,026",7.611000,35032,"2,686,706,026"
392,597,Titanic,1997-12-18,194,"200,000,000","2,264,162,353",7.901000,28068,"2,064,162,353"
8509,140607,Star Wars: The Force Awakens,2015-12-15,136,"245,000,000","2,068,223,624",7.237000,21135,"1,823,223,624"
9139,299536,Avengers: Infinity War,2018-04-25,149,"300,000,000","2,052,415,039",8.247000,33398,"1,752,415,039"
8484,135397,Jurassic World,2015-06-06,124,"150,000,000","1,671,537,444",6.702000,22114,"1,521,537,444"
3070,10733,The Alamo,2004-04-07,137,"107,000,000","25,819,961",5.800000,391,"-81,180,039"
778,1408,Cutthroat Island,1995-12-22,124,"102,000,000","16,000,000",5.803000,663,"-86,000,000"
9158,302429,Strange Magic,2015-01-23,99,"100,000,000","13,603,453",7.300000,609,"-86,396,547"
3632,11692,The Adventures of Pluto Nash,2002-08-16,95,"100,000,000","7,103,973",4.381000,565,"-92,896,027"
7541,50321,Mars Needs Moms,2011-03-09,88,"150,000,000","39,000,000",6.011000,835,"-111,000,000"


### Data
MovieLens `ml-latest-small` movies were linked through `links.csv` to their TMDB IDs and enriched via the TMDB API. The results are stored in SQLite (`movies.db`) in five tables: `movies` (title, release date, runtime, budget, revenue, vote average, vote count), `genres`, `movie_genres`, `people` and `movie_credits` (director and cast roles).

### Coverage
- 98.8% of the unique TMDB IDs in `links.csv` were downloaded (9,620 movies).
- The remaining ~1.2% (about 113 IDs) all returned a 404 from TMDB. TMDB has no record for them, so they are genuinely unavailable rather than failed downloads, and retrying would not help.
- There are no duplicate movies (checked on title + release date).

### Genres (TMDB audience ratings)
- Weighting ratings by vote count puts War (7.46), History (7.38), Documentary (7.31), Western (7.30) and Drama (7.29) at the top, and Horror (6.62) at the bottom (TV Movie ties at 6.62).
- The weighted score is higher than the plain average in almost every genre (for example Comedy: 6.33 plain vs. 6.78 weighted). Movies with many votes tend to be rated higher than obscure ones.
- Drama is the biggest genre by far (4,474 movies, about 6.5M votes), followed by Comedy by movie count (3,918).
- Blockbuster genres attract many more votes per film: roughly 3,700 for Adventure, 3,400 for Science Fiction and 3,100 for Action, against about 1,460 for Drama, 1,330 for Comedy and 210 for Documentary.
- A movie can belong to several genres, so genre counts add up to more than the number of movies.

### People
People are ranked by total votes and no minimum-films filter is applied, so the rankings reflect how widely seen a person's films are.
- Most-voted actors: Johnny Depp (48 films, avg 6.69), Robert Downey Jr. (41, 6.60), Tom Hanks (42, 6.92), Leonardo DiCaprio (25, 7.26) and Brad Pitt (33, 7.05), each with roughly 275-286k votes.
- Most-voted directors: Steven Spielberg (32 films, 7.21), Christopher Nolan (10, 7.88), Quentin Tarantino (9, 7.87), Martin Scorsese (25, 7.35) and Ridley Scott (23, 6.83).
- Popularity and quality differ: Nolan and Tarantino have much higher average ratings than Spielberg with far fewer films, and DiCaprio has the highest average among the top actors with the fewest films.

### Budget and revenue
- TMDB stores 0 when a value is unknown, so movies with revenue 0 are excluded from the profit analysis. Otherwise they would show up as false losses equal to their budget.
- Profit is not adjusted for inflation, which favors recent releases.
- The most profitable films are mega-blockbusters: Avatar (about +$2.69B), Titanic (+$2.06B), Star Wars: The Force Awakens (+$1.82B), Avengers: Infinity War (+$1.75B) and Jurassic World (+$1.52B).